## DATA GENERATION

In [261]:
!pip install faker

In [262]:
import pandas as pd
import random
import os
from faker import Faker
from datetime import datetime
fake = Faker()
random.seed(42)
Faker.seed(42)

In [263]:
# creating files
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/cleaned", exist_ok=True)

In [264]:
# to generate customer csv
def generate_customers(n=500):
  customers = []
  customer_types = ["REGULAR", "PREMIUM", "VIP"]
  for i in range(1, n + 1):
    customer_id = f"CUST{i:04d}"
    name = fake.name()
    email = fake.email()
    registration_date = fake.date_between(
    start_date="-3y",
    end_date="today"
   )
    customer_type = random.choice(customer_types)
    customers.append({
      "customer_id": customer_id,
      "customer_name": name,
      "email": email,
      "registration_date": registration_date,
      "customer_type": customer_type
     })
  df = pd.DataFrame(customers)
  return df

In [265]:
customers_df = generate_customers()

In [266]:
# adding invalid rows
invalid_rows = customers_df.sample(frac=0.02, random_state=42).index
for i in invalid_rows:
  customers_df.loc[i, "email"] = customers_df.loc[i, "email"].replace("@", "")

In [267]:
# adding missing data
missing_rows = customers_df.sample(10, random_state=1).index
customers_df.loc[missing_rows, "email"] = None

In [268]:
# making duplicate data
duplicates = customers_df.sample(10, random_state=5)
customers_df = pd.concat([customers_df, duplicates], ignore_index=True)

In [269]:
#verifing
print(customers_df.head())
print("\nTotal Rows :", len(customers_df))
print("\nMissing Emails :", customers_df["email"].isna().sum())
print("\nDuplicate Rows :", customers_df.duplicated().sum())

  customer_id      customer_name                     email registration_date  \
0    CUST0001       Allison Hill  donaldgarcia@example.net        2026-03-15   
1    CUST0002     Javier Johnson   jesseguzman@example.net        2024-02-14   
2    CUST0003  Kimberly Robinson        lisa02@example.net        2024-10-17   
3    CUST0004   Daniel Gallagher    daviscolin@example.com        2025-05-03   
4    CUST0005     Monica Herrera       smiller@example.net        2025-05-19   

  customer_type  
0           VIP  
1       REGULAR  
2       REGULAR  
3           VIP  
4       PREMIUM  

Total Rows : 510

Missing Emails : 10

Duplicate Rows : 10


In [270]:
# saving
customers_df.to_csv(
  "data/raw/customers.csv",
  index=False
)
print("customers.csv created successfully!")

customers.csv created successfully!


###Product csv genration

In [271]:
#Create the function
def generate_products(n=500):
  categories = {
      "Electronics": [
      "Laptop", "Mouse", "Keyboard", "Monitor", "Headphones",
      "Tablet", "Speaker", "Camera", "Smartphone", "Charger"
     ],
      "Clothing": [
      "T-Shirt", "Jeans", "Jacket", "Shoes", "Cap",
      "Sweater", "Shirt", "Dress", "Shorts", "Hoodie"
     ],
      "Home": [
      "Chair", "Table", "Lamp", "Curtains", "Pillow",
      "Sofa", "Fan", "Cup", "Bottle", "Mattress"
    ],"Books": [
      "Python Book", "SQL Guide", "AI Basics", "Data Science",
      "Algorithms", "Machine Learning", "Statistics",
      "Business Analytics", "Deep Learning", "Networking" ]
    }
  products = []
  for i in range(1, n + 1):
    category = random.choice(list(categories.keys()))
    product_name = random.choice(categories[category]) + f" {i}"
    subcategory = category
    cost_price = round(random.uniform(100, 5000), 2)
    products.append({
      "product_id": f"PROD{i:04d}","product_name": product_name,"category": category,"subcategory": subcategory,
      "cost_price": cost_price})
  return pd.DataFrame(products)

In [272]:
products_df = generate_products()
products_df.head()

,product_id,product_name,category,subcategory,cost_price
0,PROD0001,Speaker 1,Electronics,Electronics,4373.62
1,PROD0002,Data Science 2,Books,Books,1077.91
2,PROD0003,Machine Learning 3,Books,Books,1595.44
3,PROD0004,Shoes 4,Clothing,Clothing,215.96
4,PROD0005,Shirt 5,Clothing,Clothing,1708.48


In [273]:
#Add Extra Spaces (around 5%)
space_rows = products_df.sample(frac=0.05, random_state=42).index
products_df.loc[space_rows, "product_name"] = ( "  " + products_df.loc[space_rows, "product_name"] + "   ")

In [274]:
#Add Mixed Case Names (around 5%)
mixed_case_rows = products_df.sample(frac=0.05, random_state=7).index
products_df.loc[mixed_case_rows, "product_name"] = (products_df.loc[mixed_case_rows, "product_name"]
    .str.swapcase()
)

In [275]:
#Add Duplicate Products
duplicates = products_df.sample(10, random_state=15)
products_df = pd.concat(
    [products_df, duplicates],ignore_index=True
)

In [276]:
#Verify
print(products_df.head())
print("\nTotal Rows:", len(products_df))
print("\nDuplicate Rows:", products_df.duplicated().sum())

  product_id        product_name     category  subcategory  cost_price
0   PROD0001           Speaker 1  Electronics  Electronics     4373.62
1   PROD0002      Data Science 2        Books        Books     1077.91
2   PROD0003  Machine Learning 3        Books        Books     1595.44
3   PROD0004             Shoes 4     Clothing     Clothing      215.96
4   PROD0005             Shirt 5     Clothing     Clothing     1708.48

Total Rows: 510

Duplicate Rows: 10


In [277]:
products_df.to_csv(
    "data/raw/products.csv",
    index=False
)
print("products.csv created successfully!")

products.csv created successfully!


###Order csv

In [278]:
def generate_orders(n=500):
    statuses = ["PLACED","SHIPPED","DELIVERED","CANCELLED", "RETURNED"]
    regions = ["NORTH","SOUTH","EAST","WEST"]
    orders = []
    for i in range(1, n + 1):
        customer_id = f"CUST{random.randint(1,500):04d}"
        order_date = fake.date_time_between(
            start_date="-2y",
            end_date="now"
        )
        orders.append({"order_id": f"ORD{i:04d}",
            "customer_id": customer_id,
            "order_date": order_date.strftime("%Y-%m-%d %H:%M:%S"),
            "status": random.choice(statuses),
            "region_code": random.choice(regions)})
    return pd.DataFrame(orders)

In [279]:
orders_df = generate_orders()
orders_df.head()

,order_id,customer_id,order_date,status,region_code
0,ORD0001,CUST0318,2025-07-14 01:51:00,CANCELLED,SOUTH
1,ORD0002,CUST0421,2025-11-08 07:51:21,SHIPPED,NORTH
2,ORD0003,CUST0294,2026-03-23 16:30:19,PLACED,SOUTH
3,ORD0004,CUST0010,2026-02-21 09:58:04,CANCELLED,EAST
4,ORD0005,CUST0215,2025-10-03 19:04:43,SHIPPED,WEST


In [280]:
#Add 5% NULL Customer IDs
null_rows = orders_df.sample(frac=0.05, random_state=42).index
orders_df.loc[null_rows, "customer_id"] = None

In [281]:
#Wrong Date Format
wrong_date_rows = orders_df.sample(frac=0.05, random_state=10).index
for i in wrong_date_rows:
    date = pd.to_datetime(orders_df.loc[i, "order_date"])
    orders_df.loc[i, "order_date"] = date.strftime("%d-%m-%Y %H:%M:%S")

In [282]:
#Future Dates
future_rows = orders_df.sample(frac=0.02, random_state=5).index
for i in future_rows:
    future_date = datetime.now() + pd.Timedelta(days=random.randint(30,180))
    orders_df.loc[i, "order_date"] = future_date.strftime("%Y-%m-%d %H:%M:%S")

In [283]:
#Duplicate Rows
duplicates = orders_df.sample(10, random_state=15)
orders_df = pd.concat(
    [orders_df, duplicates],
    ignore_index=True
)

In [284]:
# verification
(orders_df.head())
print("\nTotal Rows:", len(orders_df))
print("\nMissing Customer IDs:", orders_df["customer_id"].isna().sum())
print("\nDuplicate Order IDs:", orders_df.duplicated(subset=["order_id"]).sum())


Total Rows: 510

Missing Customer IDs: 26

Duplicate Order IDs: 10


In [285]:
orders_df.to_csv(
    "data/raw/orders.csv",
    index=False
)
print("orders.csv created successfully!")

orders.csv created successfully!


###Order item csv

In [286]:
def generate_order_items(n=1500):
    items = []
    for i in range(1, n + 1):
        item_id = f"ITEM{i:05d}"
        order_id = f"ORD{random.randint(1,500):04d}"
        product_id = f"PROD{random.randint(1,500):04d}"
        quantity = random.randint(1,5)
        unit_price = round(random.uniform(100,5000),2)
        discount = random.randint(0,50)
        items.append({
            "item_id": item_id,
            "order_id": order_id,
            "product_id": product_id,
            "quantity": quantity,
            "unit_price": unit_price,
            "discount_percent": discount
        })
    return pd.DataFrame(items)

In [287]:
order_items_df = generate_order_items()
order_items_df.head()

,item_id,order_id,product_id,quantity,unit_price,discount_percent
0,ITEM00001,ORD0159,PROD0090,4,286.02,3
1,ITEM00002,ORD0120,PROD0309,2,197.65,30
2,ITEM00003,ORD0001,PROD0170,5,1087.18,8
3,ITEM00004,ORD0175,PROD0368,2,4602.03,20
4,ITEM00005,ORD0031,PROD0012,2,2974.38,9


In [288]:
#Add Negative Quantity (3%)
negative_rows = order_items_df.sample(frac=0.03, random_state=42).index
order_items_df.loc[negative_rows, "quantity"] *= -1

In [289]:
#Add Invalid Order IDs (2%)
invalid_order_rows = order_items_df.sample(frac=0.02, random_state=7).index
for i in invalid_order_rows:
    order_items_df.loc[i, "order_id"] = f"ORD{random.randint(600,700):04d}"

In [290]:
#Add Discounts Greater Than 100%
discount_rows = order_items_df.sample(frac=0.02, random_state=10).index
order_items_df.loc[discount_rows, "discount_percent"] = random.randint(101,150)

In [291]:
#Add Duplicate Rows
duplicates = order_items_df.sample(20, random_state=15)
order_items_df = pd.concat(
    [order_items_df, duplicates],
    ignore_index=True
)

In [292]:
order_items_df.to_csv(
    "data/raw/order_items.csv",index=False
)
print("order_items.csv created successfully!")

order_items.csv created successfully!


In [293]:
#verifing
print(order_items_df.head())
print("\nTotal Rows:", len(order_items_df))
print("\nNegative Quantity:",(order_items_df["quantity"] < 0).sum())
print("\nDiscount >100:",(order_items_df["discount_percent"] > 100).sum())
print("\nDuplicate Item IDs:",
order_items_df.duplicated(subset=["item_id"]).sum())

     item_id order_id product_id  quantity  unit_price  discount_percent
0  ITEM00001  ORD0159   PROD0090         4      286.02                 3
1  ITEM00002  ORD0120   PROD0309         2      197.65                30
2  ITEM00003  ORD0001   PROD0170         5     1087.18                 8
3  ITEM00004  ORD0175   PROD0368         2     4602.03                20
4  ITEM00005  ORD0031   PROD0012         2     2974.38                 9

Total Rows: 1520

Negative Quantity: 45

Discount >100: 30

Duplicate Item IDs: 20


In [294]:
import pandas as pd
for file in ["customers.csv","products.csv","orders.csv","order_items.csv"]:
    df = pd.read_csv(f"data/raw/{file}")
    print(f"{file}: {df.shape}")

customers.csv: (510, 5)
products.csv: (510, 5)
orders.csv: (510, 5)
order_items.csv: (1520, 6)


## DATA CLEANING

In [295]:
#LOAD LIBRARIES
import pandas as pd
import numpy as np
import os
os.makedirs("data/cleaned", exist_ok=True)

In [296]:
#load csv files
customers = pd.read_csv("data/raw/customers.csv")
products = pd.read_csv("data/raw/products.csv")
orders = pd.read_csv("data/raw/orders.csv")
order_items = pd.read_csv("data/raw/order_items.csv")
print("Customers:", customers.shape)
print("Products:", products.shape)
print("Orders:", orders.shape)
print("Order Items:", order_items.shape)

Customers: (510, 5)
Products: (510, 5)
Orders: (510, 5)
Order Items: (1520, 6)


In [297]:
# cleaning customer csv
print("Before Cleaning:", customers.shape)
# Remove duplicates
customers = customers.drop_duplicates()
# Fill missing emails
customers["email"] = customers["email"].fillna(
    customers["customer_name"]
    .str.lower()
    .str.replace(" ", ".", regex=False) + "@example.com"
)
# Fix invalid emails
mask = ~customers["email"].str.contains("@", na=False)
customers.loc[mask, "email"] = (
    customers.loc[mask, "customer_name"]
    .str.lower()
    .str.replace(" ", ".", regex=False)+ "@example.com"
)
# Convert registration date
customers["registration_date"] = pd.to_datetime(
    customers["registration_date"],
    errors="coerce"
)
print("After Cleaning:", customers.shape)
customers.head()

Before Cleaning: (510, 5)
After Cleaning: (500, 5)


,customer_id,customer_name,email,registration_date,customer_type
0,CUST0001,Allison Hill,donaldgarcia@example.net,2026-03-15,VIP
1,CUST0002,Javier Johnson,jesseguzman@example.net,2024-02-14,REGULAR
2,CUST0003,Kimberly Robinson,lisa02@example.net,2024-10-17,REGULAR
3,CUST0004,Daniel Gallagher,daviscolin@example.com,2025-05-03,VIP
4,CUST0005,Monica Herrera,smiller@example.net,2025-05-19,PREMIUM


In [298]:
# cleaning product csv
print("Before Cleaning:", products.shape)
# Remove duplicates
products = products.drop_duplicates()
# Remove extra spaces
products["product_name"] = products["product_name"].str.strip()
# Convert to Title Case
products["product_name"] = products["product_name"].str.title()
# Ensure cost price is numeric
products["cost_price"] = pd.to_numeric(
    products["cost_price"],
    errors="coerce"
)
# Remove invalid prices
products = products[products["cost_price"] > 0]
print("After Cleaning:", products.shape)
products.head()

Before Cleaning: (510, 5)
After Cleaning: (500, 5)


,product_id,product_name,category,subcategory,cost_price
0,PROD0001,Speaker 1,Electronics,Electronics,4373.62
1,PROD0002,Data Science 2,Books,Books,1077.91
2,PROD0003,Machine Learning 3,Books,Books,1595.44
3,PROD0004,Shoes 4,Clothing,Clothing,215.96
4,PROD0005,Shirt 5,Clothing,Clothing,1708.48


In [299]:
# cleaning order csv
print("Before Cleaning:", orders.shape)
# Remove duplicates
orders = orders.drop_duplicates()
# Remove rows with missing customer IDs
orders = orders.dropna(subset=["customer_id"])
# Convert dates
orders["order_date"] = pd.to_datetime(
    orders["order_date"],
    errors="coerce",
    dayfirst=False
)
# Remove invalid dates
orders = orders.dropna(subset=["order_date"])
# Remove future dates
orders = orders[ orders["order_date"] <= pd.Timestamp.today()]
print("After Cleaning:", orders.shape)
orders.head()

Before Cleaning: (510, 5)
After Cleaning: (443, 5)


,order_id,customer_id,order_date,status,region_code
0,ORD0001,CUST0318,2025-07-14 01:51:00,CANCELLED,SOUTH
1,ORD0002,CUST0421,2025-11-08 07:51:21,SHIPPED,NORTH
2,ORD0003,CUST0294,2026-03-23 16:30:19,PLACED,SOUTH
3,ORD0004,CUST0010,2026-02-21 09:58:04,CANCELLED,EAST
4,ORD0005,CUST0215,2025-10-03 19:04:43,SHIPPED,WEST


In [300]:
#to clean order item
print("Before Cleaning:", order_items.shape)
# Remove duplicates
order_items = order_items.drop_duplicates()
# Remove negative quantity
order_items = order_items[order_items["quantity"] > 0]
# Remove invalid discounts
order_items = order_items[order_items["discount_percent"] <= 100]
# Numeric conversion
order_items["unit_price"] = pd.to_numeric(
    order_items["unit_price"],
    errors="coerce"
)
order_items = order_items.dropna(subset=["unit_price"])
print("After Cleaning:", order_items.shape)
order_items.head()

Before Cleaning: (1520, 6)
After Cleaning: (1427, 6)


,item_id,order_id,product_id,quantity,unit_price,discount_percent
0,ITEM00001,ORD0159,PROD0090,4,286.02,3
1,ITEM00002,ORD0120,PROD0309,2,197.65,30
2,ITEM00003,ORD0001,PROD0170,5,1087.18,8
3,ITEM00004,ORD0175,PROD0368,2,4602.03,20
4,ITEM00005,ORD0031,PROD0012,2,2974.38,9


In [301]:
# to ensure that that every order_id in order_items exists in order
print("Before:", len(order_items))
order_items = order_items[order_items["order_id"].isin(orders["order_id"])]
print("After:", len(order_items))

Before: 1427
After: 1246


In [302]:
# to save all cleaned files
customers.to_csv(
    "data/cleaned/customers_clean.csv",index=False
)
products.to_csv(
    "data/cleaned/products_clean.csv",index=False
)
orders.to_csv(
    "data/cleaned/orders_clean.csv",index=False
)
order_items.to_csv(
    "data/cleaned/order_items_clean.csv",index=False
)
print("All cleaned CSV files saved successfully!")

All cleaned CSV files saved successfully!


In [303]:
print("\nFinal Dataset Shapes")
print("Customers:", customers.shape)
print("Products:", products.shape)
print("Orders:", orders.shape)
print("Order Items:", order_items.shape)


Final Dataset Shapes
Customers: (500, 5)
Products: (500, 5)
Orders: (443, 5)
Order Items: (1246, 6)


# load clean data into SQLite

In [304]:
# import libraries
import sqlite3
import pandas as pd

In [305]:
# connent database with sqlite
conn = sqlite3.connect("ecommerce.db")
cursor = conn.cursor()
print("Database Created Successfully!")

Database Created Successfully!


In [306]:
#load clean csv
customers = pd.read_csv("data/cleaned/customers_clean.csv")
products = pd.read_csv("data/cleaned/products_clean.csv")
orders = pd.read_csv("data/cleaned/orders_clean.csv")
order_items = pd.read_csv("data/cleaned/order_items_clean.csv")

In [307]:
# creating tables
cursor.executescript("""
-- Remove existing tables (if any) to avoid errors
DROP TABLE IF EXISTS order_items;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers;

-- Create Customers table
CREATE TABLE customers(
    customer_id TEXT PRIMARY KEY,      -- Unique customer ID
    customer_name TEXT,                -- Customer name
    email TEXT,                        -- Customer email
    registration_date DATE,            -- Registration date
    customer_type TEXT                 -- Customer type
);

-- Create Products table
CREATE TABLE products(
    product_id TEXT PRIMARY KEY,       -- Unique product ID
    product_name TEXT,                 -- Product name
    category TEXT,                     -- Product category
    subcategory TEXT,                  -- Product subcategory
    cost_price REAL                    -- Product cost price
);

-- Create Orders table
CREATE TABLE orders(
    order_id TEXT PRIMARY KEY,         -- Unique order ID
    customer_id TEXT,                  -- Customer who placed the order
    order_date DATE,                   -- Order date
    status TEXT,                       -- Order status
    region_code TEXT,                  -- Region code
    FOREIGN KEY(customer_id) REFERENCES customers(customer_id)
);

-- Create Order Items table
CREATE TABLE order_items(
    item_id TEXT PRIMARY KEY,          -- Unique item ID
    order_id TEXT,                     -- Order ID
    product_id TEXT,                   -- Product ID
    quantity INTEGER,                  -- Quantity ordered
    unit_price REAL,                   -- Selling price per unit
    discount_percent REAL,             -- Discount percentage
    FOREIGN KEY(order_id) REFERENCES orders(order_id),
    FOREIGN KEY(product_id) REFERENCES products(product_id)
);

""")

# Save the changes
conn.commit()

# Confirmation message
print("Tables Created Successfully!")

Tables Created Successfully!


In [308]:
# insert data
customers.to_sql("customers",conn,if_exists="append",index=False)
products.to_sql("products",conn,if_exists="append",index=False)
orders.to_sql( "orders",conn,if_exists="append",index=False)
order_items.to_sql("order_items",conn,if_exists="append",index=False)
print("Data Inserted Successfully!")

Data Inserted Successfully!


In [309]:
# verifinf row count
tables = ["customers","products","orders","order_items"]
for table in tables:
    count = pd.read_sql(
        f"SELECT COUNT(*) AS Total FROM {table}",
        conn
    )
    print(table)
    print(count)
    print("-"*30)

customers
   Total
0    500
------------------------------
products
   Total
0    500
------------------------------
orders
   Total
0    443
------------------------------
order_items
   Total
0   1246
------------------------------


In [310]:
#  query to display order details with customer names
query = """
SELECT
    o.order_id,
    c.customer_name,
    o.status,
    o.order_date
FROM orders o
JOIN customers c
ON o.customer_id = c.customer_id
LIMIT 10;
"""
# Execute the query and display the first 10 records
pd.read_sql(query, conn)

,order_id,customer_name,status,order_date
0,ORD0001,Patricia Young PhD,CANCELLED,2025-07-14 01:51:00
1,ORD0002,Susan Powers,SHIPPED,2025-11-08 07:51:21
2,ORD0003,Nicholas Hahn,PLACED,2026-03-23 16:30:19
3,ORD0004,Thomas Bradley,CANCELLED,2026-02-21 09:58:04
4,ORD0005,Diana Hays,SHIPPED,2025-10-03 19:04:43
5,ORD0006,Raymond Gonzalez DVM,SHIPPED,2024-11-10 09:35:03
6,ORD0007,Travis Hobbs,RETURNED,2025-07-22 19:41:31
7,ORD0008,Sheryl Acosta,PLACED,2025-06-03 05:01:02
8,ORD0009,Dale Edwards,SHIPPED,2024-11-20 02:46:55
9,ORD0011,Robert Monroe,SHIPPED,2025-11-05 22:36:44


## SQL analytics

### Basic SQL

1. Total revenue per category (revenue = quantity × unit_price × (1 - discount_percent/100))

In [311]:
query="""
-- Revenue by category
SELECT p.category,
SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)) AS total_revenue
FROM order_items oi
JOIN products p ON oi.product_id=p.product_id
GROUP BY p.category;
"""

pd.read_sql(query,conn)

,category,total_revenue
0,Books,1.829503e+06
1,Clothing,1.667340e+06
2,Electronics,1.627745e+06
3,Home,1.957521e+06


2. Top 10 customers by total order value

In [312]:
query="""
-- Top 10 customers
SELECT o.customer_id,
SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)) total_value
FROM orders o
JOIN order_items oi ON o.order_id=oi.order_id
GROUP BY o.customer_id
ORDER BY total_value DESC
LIMIT 10;
"""
pd.read_sql(query,conn)

,customer_id,total_value
0,CUST0294,119890.5120
1,CUST0499,98679.7430
2,CUST0154,95481.6814
3,CUST0086,93794.1648
4,CUST0480,90421.2725
5,CUST0199,80984.7606
6,CUST0290,73545.6437
7,CUST0391,73266.7942
8,CUST0163,68018.2081
9,CUST0254,65487.2140


3. Month-wise order count for the last 12 months

In [313]:
query="""
-- Orders in last 12 months
SELECT strftime('%Y-%m',order_date) month,
COUNT(*) order_count
FROM orders
WHERE order_date>=date('now','-12 months')
GROUP BY month
ORDER BY month;
"""
pd.read_sql(query,conn)

,month,order_count
0,2025-07,16
1,2025-08,14
2,2025-09,19
3,2025-10,21
4,2025-11,22
5,2025-12,20
6,2026-01,12
7,2026-02,19
8,2026-03,18
9,2026-04,15


### Intermediate Queries

4. Find customers who placed orders but never had any item delivered

In [314]:
query="""
-- Never delivered
SELECT DISTINCT customer_id
FROM orders
WHERE customer_id NOT IN(
SELECT customer_id
FROM orders
WHERE status='DELIVERED'
);
"""
pd.read_sql(query,conn)

,customer_id
0,CUST0318
1,CUST0421
2,CUST0010
3,CUST0215
4,CUST0354
...,...
217,CUST0144
218,CUST0018
219,CUST0103
220,CUST0323


5. Products that were ordered but had more returns than purchases

In [315]:
query="""
-- Returns > Purchases
SELECT p.product_name,
SUM(CASE WHEN oi.quantity<0 THEN ABS(oi.quantity) ELSE 0 END) returns,
SUM(CASE WHEN oi.quantity>0 THEN oi.quantity ELSE 0 END) purchases
FROM order_items oi
JOIN products p ON oi.product_id=p.product_id
GROUP BY p.product_name
HAVING returns>purchases;
"""
pd.read_sql(query,conn)

,product_name,returns,purchases


6. Calculate the return rate (returned items / total items) per category

In [316]:
query="""
-- Return rate
SELECT p.category,
ROUND(
SUM(CASE WHEN oi.quantity<0 THEN ABS(oi.quantity) ELSE 0 END)*100.0/
SUM(ABS(oi.quantity)),2
) return_rate
FROM order_items oi
JOIN products p ON oi.product_id=p.product_id
GROUP BY p.category;
"""
pd.read_sql(query,conn)

,category,return_rate
0,Books,0.0
1,Clothing,0.0
2,Electronics,0.0
3,Home,0.0


### Advanced Queries (Window Functions, CTEs, Subqueries)

7. Running Totals with Window Functions

In [317]:
query="""
WITH daily AS(
SELECT o.region_code,
date(o.order_date) order_date,
SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)) daily_revenue
FROM orders o
JOIN order_items oi ON o.order_id=oi.order_id
GROUP BY o.region_code,date(o.order_date)
)
SELECT *,
SUM(daily_revenue) OVER(PARTITION BY region_code ORDER BY order_date) running_total
FROM daily;
"""
pd.read_sql(query,conn)

,region_code,order_date,daily_revenue,running_total
0,EAST,2024-07-31,9196.0764,9.196076e+03
1,EAST,2024-08-08,1263.8934,1.045997e+04
2,EAST,2024-08-09,43300.2220,5.376019e+04
3,EAST,2024-08-12,23217.0833,7.697728e+04
4,EAST,2024-09-08,3957.6240,8.093490e+04
...,...,...,...,...
375,WEST,2026-05-23,16270.4746,1.606056e+06
376,WEST,2026-05-30,2269.3328,1.608326e+06
377,WEST,2026-06-04,11261.1409,1.619587e+06
378,WEST,2026-06-11,22404.1603,1.641991e+06


8. Ranking with DENSE_RANK

In [318]:
query="""
SELECT p.category,p.product_name,
SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)) total_revenue,
DENSE_RANK() OVER(
PARTITION BY p.category
ORDER BY SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)) DESC
) rank_in_category
FROM order_items oi
JOIN products p ON oi.product_id=p.product_id
GROUP BY p.category,p.product_name;
"""
pd.read_sql(query,conn)

,category,product_name,total_revenue,rank_in_category
0,Books,Ai Basics 86,49143.9452,1
1,Books,Sql Guide 235,48842.3948,2
2,Books,Deep Learning 475,41172.1427,3
3,Books,Machine Learning 486,39740.8926,4
4,Books,Networking 495,39703.8666,5
...,...,...,...,...
458,Home,Chair 111,1554.8760,125
459,Home,Curtains 457,1314.0388,126
460,Home,Mattress 418,413.7216,127
461,Home,Lamp 449,406.6816,128


9. LAG/LEAD Analysis

In [319]:
query="""
SELECT customer_id,
order_date,
LAG(order_date) OVER(PARTITION BY customer_id ORDER BY order_date) previous_order_date,
julianday(order_date)-julianday(LAG(order_date) OVER(PARTITION BY customer_id ORDER BY order_date)) days_gap
FROM orders;
"""
pd.read_sql(query,conn)

,customer_id,order_date,previous_order_date,days_gap
0,CUST0005,2026-01-22 01:38:38,None,NaN
1,CUST0005,2026-03-24 17:11:27,2026-01-22 01:38:38,61.647789
2,CUST0006,2025-06-12 10:24:18,None,NaN
3,CUST0007,2025-01-13 15:12:09,None,NaN
4,CUST0007,2026-03-23 15:53:07,2025-01-13 15:12:09,434.028449
...,...,...,...,...
438,CUST0496,2024-12-26 21:20:15,None,NaN
439,CUST0498,2026-03-16 18:35:09,None,NaN
440,CUST0499,2024-11-11 15:56:54,None,NaN
441,CUST0499,2025-03-06 19:22:05,2024-11-11 15:56:54,115.142488


10. CTE with Multiple Levels

In [320]:
query="""
WITH monthly AS(
SELECT o.customer_id,
strftime('%Y-%m',o.order_date) month,
SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)) revenue
FROM orders o
JOIN order_items oi ON o.order_id=oi.order_id
GROUP BY o.customer_id,month
),
segment AS(
SELECT *,
CASE
WHEN revenue>10000 THEN 'High'
WHEN revenue>=5000 THEN 'Medium'
ELSE 'Low'
END category
FROM monthly
)
SELECT month,category,COUNT(*) customer_count
FROM segment
GROUP BY month,category;
"""
pd.read_sql(query,conn)

,month,category,customer_count
0,2024-07,High,13
1,2024-07,Medium,1
2,2024-08,High,7
3,2024-08,Low,4
4,2024-08,Medium,5
...,...,...,...
64,2026-05,Medium,3
65,2026-06,High,12
66,2026-06,Medium,4
67,2026-07,High,4


11. NTILE for Segmentation

In [321]:
query="""
WITH cte AS(
SELECT o.customer_id,
SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)) total_value
FROM orders o
JOIN order_items oi ON o.order_id=oi.order_id
GROUP BY o.customer_id
)
SELECT customer_id,total_value,
NTILE(4) OVER(ORDER BY total_value DESC) quartile
FROM cte;
"""
pd.read_sql(query,conn)

,customer_id,total_value,quartile
0,CUST0294,119890.5120,1
1,CUST0499,98679.7430,1
2,CUST0154,95481.6814,1
3,CUST0086,93794.1648,1
4,CUST0480,90421.2725,1
...,...,...,...
290,CUST0305,904.4925,4
291,CUST0400,543.5904,4
292,CUST0375,505.5630,4
293,CUST0338,342.1795,4


12. Year-over-Year Comparison

In [322]:
query="""
WITH rev AS(
SELECT strftime('%Y',order_date) yr,
strftime('%m',order_date) mn,
SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)) revenue
FROM orders o
JOIN order_items oi ON o.order_id=oi.order_id
GROUP BY yr,mn
)
SELECT r1.yr,r1.mn,r1.revenue,
r2.revenue prev_year_revenue
FROM rev r1
LEFT JOIN rev r2
ON r1.mn=r2.mn AND r1.yr=r2.yr+1;
"""
pd.read_sql(query,conn)

,yr,mn,revenue,prev_year_revenue
0,2024,07,312143.3427,None
1,2024,08,227474.8333,None
2,2024,09,260044.5886,None
3,2024,10,197736.6022,None
4,2024,11,314500.5649,None
5,2024,12,378193.4263,None
6,2025,01,413850.3248,None
7,2025,02,253800.7456,None
8,2025,03,224002.9909,None
9,2025,04,359424.5853,None


13. First/Last Value Analysis

In [323]:
query="""
WITH x AS(
SELECT o.customer_id,p.category,o.order_date,
FIRST_VALUE(p.category) OVER(PARTITION BY o.customer_id ORDER BY o.order_date) first_category,
LAST_VALUE(p.category) OVER(
PARTITION BY o.customer_id
ORDER BY o.order_date
ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
) last_category
FROM orders o
JOIN order_items oi ON o.order_id=oi.order_id
JOIN products p ON oi.product_id=p.product_id
)
SELECT DISTINCT customer_id,first_category,last_category,
CASE WHEN first_category<>last_category THEN 'Yes' ELSE 'No' END category_shift
FROM x;
"""
pd.read_sql(query,conn)

,customer_id,first_category,last_category,category_shift
0,CUST0005,Electronics,Electronics,No
1,CUST0006,Electronics,Electronics,No
2,CUST0007,Home,Clothing,Yes
3,CUST0008,Home,Home,No
4,CUST0009,Electronics,Electronics,No
...,...,...,...,...
290,CUST0494,Clothing,Books,Yes
291,CUST0495,Home,Books,Yes
292,CUST0496,Books,Clothing,Yes
293,CUST0498,Clothing,Clothing,No


14. Cumulative Distribution

In [324]:
query="""
WITH rev AS(
SELECT o.customer_id,
SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)) revenue
FROM orders o
JOIN order_items oi ON o.order_id=oi.order_id
GROUP BY o.customer_id
)
SELECT customer_id,revenue,
SUM(revenue) OVER(ORDER BY revenue DESC) cumulative_revenue,
ROUND(SUM(revenue) OVER(ORDER BY revenue DESC)*100.0/
SUM(revenue) OVER(),2) cumulative_percent
FROM rev;
"""
pd.read_sql(query,conn)

,customer_id,revenue,cumulative_revenue,cumulative_percent
0,CUST0294,119890.5120,1.198905e+05,1.69
1,CUST0499,98679.7430,2.185703e+05,3.09
2,CUST0154,95481.6814,3.140519e+05,4.43
3,CUST0086,93794.1648,4.078461e+05,5.76
4,CUST0480,90421.2725,4.982674e+05,7.04
...,...,...,...,...
290,CUST0305,904.4925,7.080582e+06,99.98
291,CUST0400,543.5904,7.081126e+06,99.99
292,CUST0375,505.5630,7.081631e+06,99.99
293,CUST0338,342.1795,7.081973e+06,100.00


15. Complex CTE: Cohort Analysis

In [325]:
query="""
WITH cohort AS(
SELECT customer_id,
strftime('%Y-%m',registration_date) cohort_month
FROM customers
),
activity AS(
SELECT o.customer_id,
strftime('%Y-%m',o.order_date) order_month,
c.cohort_month
FROM orders o
JOIN cohort c ON o.customer_id=c.customer_id
)
SELECT cohort_month,order_month,
COUNT(DISTINCT customer_id) retained_customers
FROM activity
GROUP BY cohort_month,order_month;
"""
pd.read_sql(query,conn)

,cohort_month,order_month,retained_customers
0,2023-07,2024-08,1
1,2023-07,2024-10,1
2,2023-07,2024-11,2
3,2023-07,2024-12,1
4,2023-07,2025-02,1
...,...,...,...
355,2026-07,2025-07,1
356,2026-07,2025-11,1
357,2026-07,2026-02,1
358,2026-07,2026-03,1


16. Self-Join with Window Function

In [326]:
query="""
SELECT oi1.product_id product_a,
oi2.product_id product_b,
COUNT(*) times_bought_together
FROM order_items oi1
JOIN order_items oi2
ON oi1.order_id=oi2.order_id
AND oi1.product_id<oi2.product_id
GROUP BY product_a,product_b
ORDER BY times_bought_together DESC;
"""
pd.read_sql(query,conn)

,product_a,product_b,times_bought_together
0,PROD0059,PROD0089,2
1,PROD0059,PROD0352,2
2,PROD0092,PROD0293,2
3,PROD0092,PROD0369,2
4,PROD0103,PROD0305,2
...,...,...,...
1680,PROD0467,PROD0488,1
1681,PROD0477,PROD0479,1
1682,PROD0477,PROD0489,1
1683,PROD0478,PROD0495,1


# A simple command-line tool

In [327]:
import sqlite3
from datetime import datetime, timedelta

# Connect to SQLite
conn = sqlite3.connect("ecommerce.db")
cur = conn.cursor()

# User Input
report_type = input("Report Type (daily/weekly/monthly): ").lower()
start_date = input("Start Date (YYYY-MM-DD): ")
end_date = input("End Date (YYYY-MM-DD): ")

# Current Period Summary
summary_query = """
SELECT
COUNT(DISTINCT o.order_id),
IFNULL(SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)),0),
COUNT(DISTINCT o.customer_id)
FROM orders o
JOIN order_items oi ON o.order_id=oi.order_id
WHERE date(o.order_date) BETWEEN ? AND ?;
"""

cur.execute(summary_query,(start_date,end_date))
orders,revenue,customers = cur.fetchone()

print("\n========== REPORT ==========")
print("Report Type :",report_type)
print("Date Range  :",start_date,"to",end_date)
print("----------------------------")
print("Total Orders      :",orders)
print("Total Revenue     :",round(revenue,2))
print("Unique Customers  :",customers)

# Top 3 Products
top_query="""
SELECT p.product_name,
SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)) revenue
FROM order_items oi
JOIN products p ON oi.product_id=p.product_id
JOIN orders o ON oi.order_id=o.order_id
WHERE date(o.order_date) BETWEEN ? AND ?
GROUP BY p.product_name
ORDER BY revenue DESC
LIMIT 3;
"""

cur.execute(top_query,(start_date,end_date))

print("\nTop 3 Products")
print("----------------------------")
rows=cur.fetchall()
for i,row in enumerate(rows,1):
    print(f"{i}. {row[0]} - {round(row[1],2)}")

# Previous Period Calculation
fmt="%Y-%m-%d"
s=datetime.strptime(start_date,fmt)
e=datetime.strptime(end_date,fmt)

days=(e-s).days+1
prev_end=s-timedelta(days=1)
prev_start=prev_end-timedelta(days=days-1)

cur.execute(summary_query,
(str(prev_start.date()),str(prev_end.date())))
prev_orders,prev_revenue,prev_customers=cur.fetchone()

print("\nComparison With Previous Period")
print("----------------------------")

def percent_change(current,previous):
    if previous==0:
        return "N/A"
    return f"{((current-previous)/previous)*100:.2f}%"

print("Orders Change    :",percent_change(orders,prev_orders))
print("Revenue Change   :",percent_change(revenue,prev_revenue))
print("Customers Change :",percent_change(customers,prev_customers))

conn.close()

Report Type (daily/weekly/monthly): monthly
Start Date (YYYY-MM-DD): 2025-04-04
End Date (YYYY-MM-DD): 2026-04-04

========== REPORT ==========
Report Type : monthly
Date Range  : 2025-04-04 to 2026-04-04
----------------------------
Total Orders      : 217
Total Revenue     : 3597205.82
Unique Customers  : 180

Top 3 Products
----------------------------
1. Sql Guide 235 - 39613.43
2. Ai Basics 86 - 36416.26
3. Speaker 13 - 34135.58

Comparison With Previous Period
----------------------------
Orders Change    : 35.62%
Revenue Change   : 36.20%
Customers Change : 33.33%


#  Edge Case Handling

1. What happens when order_items has an order_id not in orders?

In [328]:
def test_invalid_order_id(orders, order_items):
    print("Test 1: Invalid order_id")

    invalid_orders = order_items[
        ~order_items["order_id"].isin(orders["order_id"])
    ]

    if invalid_orders.empty:
        print("PASS: No invalid order_id found.")
    else:
        print(f"FAIL: {len(invalid_orders)} invalid order_id(s) found.")
        print(invalid_orders.head())

2. What happens when discount_percent > 100?

In [333]:
def test_discount_percentage(order_items):
    print("\nTest 2: Discount > 100%")

    invalid_discount = order_items[
        order_items["discount_percent"] > 100
    ]

    if invalid_discount.empty:
        print("PASS: All discounts are valid.")
    else:
        print(f"FAIL: {len(invalid_discount)} invalid discount(s) found.")
        print(invalid_discount.head())

In [334]:
test_discount_percentage(order_items)


Test 2: Discount > 100%
PASS: All discounts are valid.


3. What happens when quantity is 0?

In [331]:
def test_zero_quantity(order_items):
    print("\nTest 3: Quantity = 0")

    zero_qty = order_items[
        order_items["quantity"] == 0
    ]

    if zero_qty.empty:
        print("PASS: No zero quantity records found.")
    else:
        print(f"FAIL: {len(zero_qty)} zero quantity record(s) found.")
        print(zero_qty.head())

In [332]:
test_zero_quantity(order_items)


Test 3: Quantity = 0
PASS: No zero quantity records found.


4. What happens when order_date is in the future?

In [330]:
def test_future_order_date(orders):
    print("\nTest 4: Future Order Dates")

    orders["order_date"] = pd.to_datetime(
        orders["order_date"],
        errors="coerce"
    )

    future_orders = orders[
        orders["order_date"] > pd.Timestamp.today()
    ]

    if future_orders.empty:
        print("PASS: No future order dates found.")
    else:
        print(f"FAIL: {len(future_orders)} future order(s) found.")
        print(future_orders.head())